# Teste do setup do TFlite

## Importar bibliotecas

In [ ]:
import tflite_runtime.interpreter as tflite
import numpy as np
from PIL import Image

In [ ]:
print("NumPy:", np.__version__)
print("Pillow:", Image.__version__)

## Baixar modelo pré-treinado


- Baixe o modelo pré-treinado MobileNetV2    

    - Um modelo pré-treinado adequado é muito importante para o sucesso da classificação de imagens em dispositivos com recursos limitados, como o Raspberry Pi.
    - O [*MobileNet*](https://github.com/tensorflow/models/tree/master/research/slim/nets/mobilenet) foi projetado para aplicações móveis e de visão embarcada, com um bom equilíbrio entre precisão e velocidade
    
    - Várias versões estão disponíveis: `MobileNetV1`, `MobileNetV2`, `MobileNetV3`.
- Vamos baixar a V2:

In [ ]:
from pathlib import Path
import tarfile
import urllib.request

# Define o diretório onde os modelos serão salvos
models_dir = Path("./models")
models_dir.mkdir(parents=True, exist_ok=True)

# Define os caminhos dos arquivos do modelo e do arquivo compactado
tflite_file = models_dir / "mobilenet_v2_1.0_224_quant.tflite"
tgz_file = models_dir / "mobilenet_v2_1.0_224_quant.tgz"
url = "https://storage.googleapis.com/download.tensorflow.org/models/tflite_11_05_08/mobilenet_v2_1.0_224_quant.tgz"

# Verifica se o arquivo .tflite já existe
if tflite_file.exists():
    print(f"{tflite_file} já existe. Pulando download e extração.")
else:
    # Se o arquivo .tgz não existe, faz o download
    if not tgz_file.exists():
        print(f"Baixando {url} para {tgz_file} ...")
        urllib.request.urlretrieve(url, tgz_file)
    else:
        print(f"{tgz_file} já existe. Pulando download.")
    try:
        # Extrai o arquivo .tgz para o diretório de modelos
        print(f"Extraindo {tgz_file} para {models_dir} ...")
        with tarfile.open(tgz_file, "r:gz") as tar:
            tar.extractall(path=models_dir)
        print("Extração concluída.")
    except Exception as e:
        print("Falha ao extrair:", e)

# Verifica novamente se o arquivo .tflite está disponível
if tflite_file.exists():
    print("Arquivo .tflite pronto:", tflite_file)
else:
    print("Arquivo .tflite não encontrado após extração.")

- Agora faça upload do rótulos (labels) das classes para o seu RPi

In [ ]:
# Instala o pacote gdown para baixar arquivos do Google Drive
!pip install gdown

# Baixa o arquivo labels.txt do Google Drive para a pasta models
!gdown --id 1k5lS1vY6L9Cw-YeIGsNFEeNiwygBJlRB -O ./models/labels.txt

print("Download concluído!")

- Liste os arquivos do diretório `models`:

In [ ]:
ls ~/Documents/TFLITE/IMG_CLASS/models

Você verá algo parecido com:

```bash
2_teste_setup.ipynb
mobilenet_v2_1.0_224_quant.ckpt.data-00000-of-00001
mobilenet_v2_1.0_224_quant.ckpt.index
mobilenet_v2_1.0_224_quant.ckpt.meta
mobilenet_v2_1.0_224_quant.tflite
mobilenet_v2_1.0_224_quant.tgz
mobilenet_v2_1.0_224_quant_eval.pbtxt
mobilenet_v2_1.0_224_quant_frozen.pb
mobilenet_v2_1.0_224_quant_info.txt
```

- No entanto, apenas precisamos apenas do modelo `mobilenet_v2_1.0_224_quant.tflite` e do arquivo `labels.txt` com os rótulos das classes.
    - Você pode apagar os outros arquivos baixados.
- O arquivo `labels.txt` contém os rótulos das **1001** classes do modelo `MobileNetV2`, que são usados para interpretar as previsões do modelo.

In [ ]:
model_path = "./models/mobilenet_v2_1.0_224_quant.tflite"

In [ ]:
# Experimente criar um interpretador TFLite
interpreter = tflite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()
print("Interpretador TFLite criado com sucesso!")